# 02. 데이터 전처리
Netflix 고객 이탈 예측을 위한 데이터 전처리

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/netflix_customer_churn.csv')
print(f'원본 데이터: {df.shape}')
df.head()

원본 데이터: (5000, 14)


,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


---
## 1. 컬럼 제거
- `customer_id`: 고유 ID, 분석 무관
- `monthly_fee`: subscription_type과 1:1 대응 (Basic=8.99, Standard=13.99, Premium=17.99) → 중복 정보

In [3]:
# 1:1 대응 확인
print('=== subscription_type과 monthly_fee 대응 확인 ===')
print(df.groupby('subscription_type')['monthly_fee'].unique())

# 제거
df = df.drop(columns=['customer_id', 'monthly_fee'])
print(f'\n제거 후: {df.shape}')
df.head()

=== subscription_type과 monthly_fee 대응 확인 ===
subscription_type
Basic        [8.99]
Premium     [17.99]
Standard    [13.99]
Name: monthly_fee, dtype: object

제거 후: (5000, 12)


,age,gender,subscription_type,watch_hours,last_login_days,region,device,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,51,Other,Basic,14.73,29,Africa,TV,1,Gift Card,1,0.49,Action
1,47,Other,Standard,0.70,19,Europe,Mobile,1,Gift Card,5,0.03,Sci-Fi
2,27,Female,Standard,16.32,10,Asia,TV,0,Crypto,2,1.48,Drama
3,53,Other,Premium,4.51,12,Oceania,TV,1,Crypto,2,0.35,Horror
4,56,Other,Standard,1.89,13,Africa,Mobile,1,Crypto,2,0.13,Action


---
## 2. 이상치 제거
- `avg_watch_time_per_day`: 하루 24시간을 초과하는 값은 물리적으로 불가능한 오류 데이터
- 해당 레코드 제거 (약 10건)

> **파생변수 실험 결과**: `inactive_user_flag`, `estimated_days`, `login_inactivity_ratio`를 실험했으나
> 원본 변수만으로도 부스팅 모델 성능(F1 0.99+)이 동일 수준으로 달성되어 최종 파이프라인에서는 제외함.
> 모델 단순화 및 다중공선성 방지 목적.

In [4]:
# 이상치 확인: 하루 24시간을 초과하는 avg_watch_time_per_day
outliers = df[df['avg_watch_time_per_day'] > 24]
print(f'=== 이상치 ({len(outliers)}건) ===')
print(f'avg_watch_time_per_day 최대값: {df["avg_watch_time_per_day"].max():.2f}시간')
print(f'24시간 초과 레코드 수: {len(outliers)}')

# 제거
before = df.shape[0]
df = df[df['avg_watch_time_per_day'] <= 24].reset_index(drop=True)
after = df.shape[0]

print(f'\n제거 전: {before}건 → 제거 후: {after}건 (-{before - after}건)')
print(f'이탈률 변화: {outliers["churned"].mean():.2%} (제거된 그룹) vs {df["churned"].mean():.2%} (잔존 그룹)')

=== 이상치 (10건) ===
avg_watch_time_per_day 최대값: 98.42시간
24시간 초과 레코드 수: 10

제거 전: 5000건 → 제거 후: 4990건 (-10건)
이탈률 변화: 0.00% (제거된 그룹) vs 50.40% (잔존 그룹)


---
## 3. 피처 / 타겟 분리

In [5]:
X = df.drop(columns=['churned'])
y = df['churned']

cat_cols = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
num_cols = [c for c in X.columns if c not in cat_cols]

print(f'X: {X.shape}, y: {y.shape}')
print(f'\n수치형 ({len(num_cols)}개): {num_cols}')
print(f'범주형 ({len(cat_cols)}개): {cat_cols}')

X: (4990, 11), y: (4990,)

수치형 (5개): ['age', 'watch_hours', 'last_login_days', 'number_of_profiles', 'avg_watch_time_per_day']
범주형 (6개): ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']


---
## 4. Train / Test 분리

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train 이탈률: {y_train.mean():.2%}')
print(f'Test  이탈률: {y_test.mean():.2%}')

Train: (3992, 11), Test: (998, 11)
Train 이탈률: 50.40%
Test  이탈률: 50.40%


---
## 5. 인코딩
모델별로 적합한 인코딩 방식이 다르므로 2가지 버전을 준비합니다.

| 인코딩 | 사용 모델 | 이유 |
|---|---|---|
| One-Hot | Logistic Regression, SVM, MLP | 숫자 크기를 계산에 쓰므로 순서 착각 방지 |
| Label | Decision Tree, RF, XGBoost, LightGBM | 분기만 하므로 크기 무관, OHE 비효율 |

In [7]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# === Label Encoding 버전 (트리 모델용) ===
label_encoders = {}
X_train_label = X_train.copy()
X_test_label = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_train_label[col] = le.fit_transform(X_train_label[col])
    X_test_label[col] = le.transform(X_test_label[col])
    label_encoders[col] = le

print(f'Label Encoding 버전: {X_train_label.shape}')
X_train_label.head()

Label Encoding 버전: (3992, 11)


,age,gender,subscription_type,watch_hours,last_login_days,region,device,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
3870,62,1,2,21.13,36,1,2,4,2,0.57,2
3270,44,0,2,3.08,58,1,3,4,1,0.05,5
1861,50,2,0,13.21,36,1,4,1,4,0.36,1
3939,31,0,1,0.57,60,4,2,1,3,0.01,0
4865,51,2,0,10.44,40,1,3,0,2,0.25,3


In [8]:
# === One-Hot Encoding 버전 (LR, SVM, MLP용) ===
X_train_ohe = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# train에 있는 컬럼 기준으로 맞추기
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

print(f'One-Hot Encoding 버전: {X_train_ohe.shape}')
X_train_ohe.head()

One-Hot Encoding 버전: (3992, 28)


,age,watch_hours,last_login_days,number_of_profiles,avg_watch_time_per_day,gender_Male,gender_Other,subscription_type_Premium,subscription_type_Standard,region_Asia,...,payment_method_Crypto,payment_method_Debit Card,payment_method_Gift Card,payment_method_PayPal,favorite_genre_Comedy,favorite_genre_Documentary,favorite_genre_Drama,favorite_genre_Horror,favorite_genre_Romance,favorite_genre_Sci-Fi
3870,62,21.13,36,2,0.57,True,False,False,True,True,...,False,False,False,True,False,True,False,False,False,False
3270,44,3.08,58,1,0.05,False,False,False,True,True,...,False,False,False,True,False,False,False,False,True,False
1861,50,13.21,36,4,0.36,False,True,False,False,True,...,True,False,False,False,True,False,False,False,False,False
3939,31,0.57,60,3,0.01,False,False,True,False,False,...,True,False,False,False,False,False,False,False,False,False
4865,51,10.44,40,2,0.25,False,True,False,False,True,...,False,False,False,False,False,False,True,False,False,False


---
## 6. 스케일링

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# OHE 버전 스케일링 (수치형 컬럼만)
X_train_ohe_scaled = X_train_ohe.copy()
X_test_ohe_scaled = X_test_ohe.copy()
X_train_ohe_scaled[num_cols] = scaler.fit_transform(X_train_ohe[num_cols])
X_test_ohe_scaled[num_cols] = scaler.transform(X_test_ohe[num_cols])

print('스케일링 완료 (OHE 버전, 수치형 컬럼만)')
X_train_ohe_scaled[num_cols].describe().round(2)

스케일링 완료 (OHE 버전, 수치형 컬럼만)


,age,watch_hours,last_login_days,number_of_profiles,avg_watch_time_per_day
count,3992.00,3992.00,3992.00,3992.00,3992.00
mean,0.00,0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-1.66,-0.99,-1.72,-1.44,-0.45
25%,-0.89,-0.70,-0.86,-0.73,-0.39
50%,0.02,-0.30,-0.01,-0.02,-0.29
75%,0.92,0.38,0.84,0.68,-0.05
max,1.69,8.47,1.70,1.39,12.32


---
## 7. 전처리 결과 저장

In [10]:
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 전처리된 데이터 저장
X_train_label.to_csv('../data/processed/X_train_label.csv', index=False)
X_test_label.to_csv('../data/processed/X_test_label.csv', index=False)
X_train_ohe_scaled.to_csv('../data/processed/X_train_ohe_scaled.csv', index=False)
X_test_ohe_scaled.to_csv('../data/processed/X_test_ohe_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# 전처리 도구 저장 (나중에 Streamlit에서 재사용)
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl')

print('=== 저장 완료 ===')
print(f'Label 버전  - Train: {X_train_label.shape}, Test: {X_test_label.shape}')
print(f'OHE 버전    - Train: {X_train_ohe_scaled.shape}, Test: {X_test_ohe_scaled.shape}')
print(f'타겟        - Train: {y_train.shape}, Test: {y_test.shape}')

=== 저장 완료 ===
Label 버전  - Train: (3992, 11), Test: (998, 11)
OHE 버전    - Train: (3992, 28), Test: (998, 28)
타겟        - Train: (3992,), Test: (998,)


---
## 전처리 요약

| 단계 | 내용 |
|---|---|
| 제거 (컬럼) | customer_id, monthly_fee (subscription_type과 1:1 대응) |
| 제거 (이상치) | avg_watch_time_per_day > 24 (10건, 물리적 불가 값) |
| 파생변수 | 실험 후 제외 — 원본 변수만으로 충분한 성능 확보 |
| 분리 | train 80% / test 20% (stratify) |
| 인코딩 | Label (트리 모델용), One-Hot (LR/SVM/MLP용) |
| 스케일링 | StandardScaler (OHE 버전 수치형 컬럼만) |

### 다음 노트북(03_modeling)에서 사용할 데이터
- **트리 모델**: `X_train_label`, `X_test_label`
- **LR, SVM, MLP**: `X_train_ohe_scaled`, `X_test_ohe_scaled`
- **CatBoost**: 원본 `X_train`, `X_test` (인코딩 불필요)